In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from xgboost.spark import SparkXGBClassifier

# Vector Assembler
#=====================================================
def build_features(df):

    df = df.withColumnRenamed("Loan_Status", "label")

    feature_cols = [
        "Age","Income","LoanAmount","Credit_Score",
        "Employment_Years","Credit_History","Has_Defaulted",
        "Dependents","Gender","Education_Level",
        "Married","Job_Type","Property_Area"
    ]

    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features",
        handleInvalid="keep"
    )

    return assembler.transform(df)

# Train and Test split
#===============================================
def split_data(df, ratio=0.8):
train_df, test_df = df.randomSplit([ratio, 1-ratio], seed=42)
return train_df, test_df

#XGBoost Model (Spark)
#================================================
def create_model():

return SparkXGBClassifier(
    features_col="features",
    label_col="label",
    prediction_col="prediction",
    probability_col="probability",
    max_depth=6,
    eta=0.1,
    num_round=100,
    objective="binary:logistic"
    )

def train_model(model, train_df):
  return model.fit(train_df)

def predict(model, test_df):
  return model.transform(test_df)

#Evaluation
#=============================================
#Accuracy, F1-score, AUC
def evaluate(predictions):

  acc = MulticlassClassificationEvaluator(
      labelCol="label",
      predictionCol="prediction",
      metricName="accuracy"
      ).evaluate(predictions)

  f1 = MulticlassClassificationEvaluator(
      labelCol="label",
      predictionCol="prediction",
      metricName="f1"
      ).evaluate(predictions)

  auc = BinaryClassificationEvaluator(
      labelCol="label",
      rawPredictionCol="rawPrediction",
      metricName="areaUnderROC"
       ).evaluate(predictions)

    return acc, f1, auc
#importance
model.get_feature_importances()


#Visualization XGBOOST
#=============================================

#Confusion Matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion_matrix(predictions):

    pdf = predictions.select("label", "prediction").toPandas()

    cm = pd.crosstab(pdf["label"], pdf["prediction"])

    fig, ax = plt.subplots()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)

    return fig


#ROC Curve
from sklearn.metrics import roc_curve, auc

def plot_roc(predictions):

    pdf = predictions.select("label", "probability").toPandas()

    y_true = predictions.select("label").toPandas()["label"]
    y_score = pdf["probability"].apply(lambda x: float(x[1]))

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots()
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    ax.plot([0,1],[0,1],'--')
    ax.set_title("ROC Curve")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend()

    return fig

#Feature Importance
def plot_feature_importance(model):

    import pandas as pd

    importance = model.get_feature_importances()

    fi = pd.DataFrame({
        "feature": list(importance.keys()),
        "importance": list(importance.values())
    }).sort_values(by="importance", ascending=False)

    fig, ax = plt.subplots()
    sns.barplot(x="importance", y="feature", data=fi, ax=ax)

    return fig
